# Check a prepared dataset

Sanity checks on the output of `scripts/prepare_data.py`. Each check is an `assert`, so the notebook fails loudly if
something is wrong. The plots at the end are for a visual check: cold poles, warm tropics, continents in the right
place.

Set `DATA_CONFIG` to the dataset to check, then run all cells.

In [ ]:
import os
from pathlib import Path


if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nanoweather.config import load_config
from nanoweather.data.datasets import ForecastPairs, WeatherData
from nanoweather.evaluation.forecasters import load_climatology
from nanoweather.processing.prepare import expand_variables


DATA_CONFIG = "configs/data.yaml"  # or experiments/configs/data_6vars.yaml
data_config = load_config(DATA_CONFIG)

## 1. Shapes, coordinates and time axis

In [ ]:
time_step = pd.Timedelta(hours=6)
EXPECTED_STEPS = {
    split: int((pd.Timestamp(end) - pd.Timestamp(start)) / time_step) + 1
    for split, (start, end) in data_config["splits"].items()
}
EXPECTED_VARIABLES = list(expand_variables(data_config))
num_variables = len(EXPECTED_VARIABLES)

splits = {
    split: WeatherData(data_config["output_dir"], split=split, store_name=data_config["store_name"])
    for split in EXPECTED_STEPS
}
for split, data in splits.items():
    assert data.states.shape == (EXPECTED_STEPS[split], num_variables, 32, 64), (split, data.states.shape)
    assert data.variables == EXPECTED_VARIABLES
    assert np.all(np.diff(data.times) == np.timedelta64(6, "h")), f"gaps in {split}"
    assert np.isfinite(data.states).all(), f"non-finite values in {split}"
    print(f"{split:5s} {data.states.shape}  {pd.Timestamp(data.times[0])} -> {pd.Timestamp(data.times[-1])}")

train = splits["train"]
assert np.allclose(train.latitude[[0, -1]], [-87.1875, 87.1875]), "latitude should run south -> north"
assert np.allclose(train.longitude[[0, -1]], [0.0, 354.375])

## 2. Physical ranges (train split, physical units)

In [ ]:
# Loose bounds for the main variables: they catch unit or variable mix-ups, not subtle errors.
EXPECTED_MEAN_RANGE = {
    "z500": (5.3e4, 5.7e4),  # m**2 s**-2, about 5.5 km x g
    "t850": (270, 285),  # K
    "t2m": (275, 282),  # K
    "u10": (-1, 2),  # m s**-1
    "v10": (-1, 1),  # m s**-1
    "msl": (1.005e5, 1.015e5),  # Pa
}
for index, name in enumerate(train.variables):
    if name not in EXPECTED_MEAN_RANGE:
        continue
    mean = train.states[:, index].mean()
    low, high = EXPECTED_MEAN_RANGE[name]
    status = "ok" if low <= mean <= high else "OUT OF RANGE"
    print(
        f"{name:5s} mean {mean:12.4g}  min {train.states[:, index].min():12.4g}  max {train.states[:, index].max():12.4g}  {status}"
    )
    assert low <= mean <= high, name

## 3. Normalization

Train split should be about mean 0 and std 1 per variable. Val and test use the train statistics, so they are close but not exact.

In [ ]:
for split, data in splits.items():
    mean = data.normalized_states.mean(axis=(0, 2, 3))
    std = data.normalized_states.std(axis=(0, 2, 3))
    print(f"{split:5s} mean {np.round(mean, 3)}  std {np.round(std, 3)}")

assert np.allclose(train.normalized_states.mean(axis=(0, 2, 3)), 0, atol=1e-3)
assert np.allclose(train.normalized_states.std(axis=(0, 2, 3)), 1, atol=1e-3)

round_trip = train.feature_extractor.denormalize(train.normalized_states[:10])
assert np.allclose(round_trip, train.states[:10], rtol=1e-5, atol=1e-2)

## 4. Static fields, latitude weights, training pairs

In [ ]:
assert train.static.shape == (2, 32, 64)
assert np.isclose(train.latitude_weights.mean(), 1.0)
assert train.latitude_weights[16] > train.latitude_weights[0], "weights should be larger near the equator"

pairs = ForecastPairs(train)
sample = pairs[0]
inputs, targets = sample["states"], sample["labels"]
assert len(pairs) == EXPECTED_STEPS["train"] - 1
assert inputs.shape == targets.shape == (num_variables, 32, 64)
assert inputs.dtype == targets.dtype
print(f"{len(pairs)} training pairs, input {tuple(inputs.shape)} {inputs.dtype}")

## 5. Climatology

In [ ]:
climatology = load_climatology(train.store_path, train.variables)
assert dict(climatology.sizes) == {"dayofyear": 366, "hour": 4, "latitude": 32, "longitude": 64}, dict(
    climatology.sizes
)
assert list(climatology.data_vars) == EXPECTED_VARIABLES

# Climatology should be close to the train mean (both are long-term averages of ERA5).
for index, name in enumerate(train.variables):
    clim_mean = float(climatology[name].mean())
    print(f"{name:5s} climatology mean {clim_mean:12.4g}  train mean {train.states[:, index].mean():12.4g}")

## 6. Visual checks

In [ ]:
t2m = train.variables.index("t2m")
z500 = train.variables.index("z500")

fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
panels = [
    (train.states[0, t2m] - 273.15, "t2m (°C), " + str(pd.Timestamp(train.times[0])), "RdYlBu_r"),
    (train.states[0, z500] / 9.80665, "z500 height (m)", "viridis"),
    (train.static[0], "land-sea mask (normalized)", "Greens"),
    (train.static[1], "orography (normalized)", "terrain"),
]
for ax, (values, title, cmap) in zip(axes.flat, panels):
    mesh = ax.pcolormesh(train.longitude, train.latitude, values, cmap=cmap)
    ax.set_title(title)
    fig.colorbar(mesh, ax=ax)
plt.show()

In [ ]:
# Global mean t2m: a clear seasonal cycle each year, peaking in northern summer.
weights = train.latitude_weights[:, None]
global_t2m = (train.states[:, t2m] * weights).mean(axis=(1, 2)) - 273.15

plt.figure(figsize=(12, 3))
plt.plot(pd.to_datetime(train.times), global_t2m, linewidth=0.5)
plt.ylabel("global mean t2m (°C)")
plt.show()